# Predicting Apps Download

# Fire up Libraries

In [11]:
import numpy as np
import pandas as pd
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from collections import Counter
import matplotlib.pyplot as plt
%matplotlib inline
from sklearn import metrics
from sklearn import tree
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from datetime import datetime
from random import randint



# Tratamento dos dados

In [2]:
products = pd.read_csv('bd1.csv')

In [3]:
products2 = pd.read_csv('bd2.csv')

In [4]:
dados = pd.read_csv('teste2.csv')

In [ ]:
products.shape

In [15]:
products.head()

,Unnamed: 0,channel0,channel1,channel2,channel3,channel4,channel5,channel6,channel7,channel8,...,ip2274,ip2275,ip2276,ip2277,ip2278,dia0,dia1,dia2,dia3,output
0,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
1,1,0,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
2,2,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,1
3,3,0,0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
4,4,0,0,0,1,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0


In [5]:
fracao = dados[dados['is_attributed']==1].shape[0]*1.0/dados[dados['is_attributed']==0].shape[0]
classe0 = dados[dados['is_attributed']==0]
classe1 = dados[dados['is_attributed']==1]
classe0b = classe0.sample(frac=fracao, random_state=200)
classe0b.shape

(9777, 8)

In [6]:
pieces = [classe0b, classe1]
bd1 = pd.concat(pieces)
bd1 = bd1.sample(frac=1, random_state=200).reset_index(drop=True)
bd1.head()

,ip,app,device,os,channel,click_time,attributed_time,is_attributed
0,65598,3,1,8,280,2017-11-09 00:46:05,NaN,0
1,43279,14,1,10,134,2017-11-08 03:50:41,NaN,0
2,106223,35,1,19,21,2017-11-08 08:04:43,2017-11-08 08:09:59,1
3,112468,35,1,19,21,2017-11-09 10:09:13,2017-11-09 11:25:28,1
4,123733,11,1,19,173,2017-11-09 07:43:57,NaN,0


In [7]:
fracao = bd1.shape[0]*1.0/dados.shape[0]
bd2 = dados.sample(frac=fracao, random_state=200)
bd2 = bd2.sample(frac=1, random_state=200).reset_index(drop=True)
bd2.shape

(19554, 8)

## BD1

In [5]:
train_data = products.sample(frac=0.8, random_state=200)
test_data  = products.drop(train_data.index)
print(train_data.shape, test_data.shape)

((15643, 2998), (3911, 2998))


## BD2

In [13]:
train_data = products2.sample(frac=0.8, random_state=200)
test_data  = products2.drop(train_data.index)
print(train_data.shape, test_data.shape)

((15643, 3623), (3911, 3623))


# Regressão Logística

## BD1

In [12]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [14]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

In [15]:
x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

### Cross Validation

In [16]:
t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.87814679099
C= 1 , t= 0.3 , F1: 0.885980832015
C= 10 , t= 0.3 , F1: 0.886843086251
C= 0.1 , t= 0.4 , F1: 0.891880983955
C= 1 , t= 0.4 , F1: 0.893595489623
C= 10 , t= 0.4 , F1: 0.893144752597
C= 0.1 , t= 0.5 , F1: 0.885356828614
C= 1 , t= 0.5 , F1: 0.893869253252
C= 10 , t= 0.5 , F1: 0.894156520397


In [47]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)
    
classifier = LogisticRegression(C=10)
classifier.fit(x_train, y_train.ravel())

LogisticRegression(C=10, class_weight=None, dual=False, fit_intercept=True,
          intercept_scaling=1, max_iter=100, multi_class='ovr', n_jobs=1,
          penalty='l2', random_state=None, solver='liblinear', tol=0.0001,
          verbose=0, warm_start=False)

### Evaluate the model

In [48]:
x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['output'].values

# predict class labels for the test set
predicted = classifier.predict(x_test)

In [46]:
predicted

array([1, 1, 0, ..., 1, 1, 1])

In [49]:
# generate class probabilities
probs = classifier.predict_proba(x_test)
print probs

[[ 0.00172197  0.99827803]
 [ 0.0020545   0.9979455 ]
 [ 0.96568839  0.03431161]
 ..., 
 [ 0.00122128  0.99877872]
 [ 0.00172197  0.99827803]
 [ 0.00343218  0.99656782]]


In [50]:
pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i][1]) > 0.5):
        pred[i] = 1
    else:
        pred[i] = 0

In [51]:
# generate evaluation metrics
print metrics.accuracy_score(y_test, pred)
print metrics.roc_auc_score(y_test, probs[:, 1])
print metrics.precision_score(y_test, pred)
print metrics.recall_score(y_test, pred)
print metrics.f1_score(y_test, pred)

0.913321401176
0.953571932908
0.961714285714
0.860869565217
0.908502024291


# Seleção de features

In [17]:
my_features = []

#for i in range(bd1['channel'].unique().shape[0]):
 #   my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [18]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.851983576545
C= 1 , t= 0.3 , F1: 0.877651897566
C= 10 , t= 0.3 , F1: 0.882635089247
C= 0.1 , t= 0.4 , F1: 0.883656646514
C= 1 , t= 0.4 , F1: 0.88659884489
C= 10 , t= 0.4 , F1: 0.889106842905
C= 0.1 , t= 0.5 , F1: 0.875623828033
C= 1 , t= 0.5 , F1: 0.890524447403
C= 10 , t= 0.5 , F1: 0.890412168904


In [19]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
#for i in range(bd1['app'].unique().shape[0]):
 #   my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [20]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.85091751123
C= 1 , t= 0.3 , F1: 0.874177366222
C= 10 , t= 0.3 , F1: 0.876610017266
C= 0.1 , t= 0.4 , F1: 0.880601014648
C= 1 , t= 0.4 , F1: 0.885044131165
C= 10 , t= 0.4 , F1: 0.883710877808
C= 0.1 , t= 0.5 , F1: 0.86198219581
C= 1 , t= 0.5 , F1: 0.884024572286
C= 10 , t= 0.5 , F1: 0.885721125821


In [21]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd1['device'].unique().shape[0]):
 #   my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [22]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.875665878143
C= 1 , t= 0.3 , F1: 0.882931347933
C= 10 , t= 0.3 , F1: 0.884619037303
C= 0.1 , t= 0.4 , F1: 0.890578468227
C= 1 , t= 0.4 , F1: 0.89256024557
C= 10 , t= 0.4 , F1: 0.891091415849
C= 0.1 , t= 0.5 , F1: 0.88708184004
C= 1 , t= 0.5 , F1: 0.893594442939
C= 10 , t= 0.5 , F1: 0.892825158762


In [23]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
 #   my_features.append('os' + str(i))
    
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [24]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.876673247412
C= 1 , t= 0.3 , F1: 0.885562681739
C= 10 , t= 0.3 , F1: 0.884831396453
C= 0.1 , t= 0.4 , F1: 0.889661079514
C= 1 , t= 0.4 , F1: 0.89245443585
C= 10 , t= 0.4 , F1: 0.88934790429
C= 0.1 , t= 0.5 , F1: 0.884885409395
C= 1 , t= 0.5 , F1: 0.893749065788
C= 10 , t= 0.5 , F1: 0.896307883126


In [25]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [26]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.878120764983
C= 1 , t= 0.3 , F1: 0.885095854201
C= 10 , t= 0.3 , F1: 0.886847833623
C= 0.1 , t= 0.4 , F1: 0.890841875148
C= 1 , t= 0.4 , F1: 0.894396840459
C= 10 , t= 0.4 , F1: 0.893904057254
C= 0.1 , t= 0.5 , F1: 0.885234408539
C= 1 , t= 0.5 , F1: 0.89291751107
C= 10 , t= 0.5 , F1: 0.896377019123


## Seleção gulosa: n=4

In [29]:
my_features = []

#for i in range(bd1['channel'].unique().shape[0]):
#    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [30]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.851815102835
C= 1 , t= 0.3 , F1: 0.877197022301
C= 10 , t= 0.3 , F1: 0.881854947921
C= 0.1 , t= 0.4 , F1: 0.883921618635
C= 1 , t= 0.4 , F1: 0.886632085074
C= 10 , t= 0.4 , F1: 0.888702089749
C= 0.1 , t= 0.5 , F1: 0.874963377712
C= 1 , t= 0.5 , F1: 0.889568063325
C= 10 , t= 0.5 , F1: 0.890908804573


In [31]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
#for i in range(bd1['app'].unique().shape[0]):
#    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [32]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.851674249698
C= 1 , t= 0.3 , F1: 0.875662857503
C= 10 , t= 0.3 , F1: 0.87797671394
C= 0.1 , t= 0.4 , F1: 0.880580974512
C= 1 , t= 0.4 , F1: 0.885941466245
C= 10 , t= 0.4 , F1: 0.886160776216
C= 0.1 , t= 0.5 , F1: 0.861681405785
C= 1 , t= 0.5 , F1: 0.884703159156
C= 10 , t= 0.5 , F1: 0.888265599177


In [33]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd1['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [34]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.875281779046
C= 1 , t= 0.3 , F1: 0.883282276637
C= 10 , t= 0.3 , F1: 0.884986064312
C= 0.1 , t= 0.4 , F1: 0.890391542077
C= 1 , t= 0.4 , F1: 0.892146625897
C= 10 , t= 0.4 , F1: 0.891459744224
C= 0.1 , t= 0.5 , F1: 0.886957185262
C= 1 , t= 0.5 , F1: 0.892856088752
C= 10 , t= 0.5 , F1: 0.895619637097


In [35]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [36]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.876967353753
C= 1 , t= 0.3 , F1: 0.886336210964
C= 10 , t= 0.3 , F1: 0.887299957975
C= 0.1 , t= 0.4 , F1: 0.889484964941
C= 1 , t= 0.4 , F1: 0.892062569916
C= 10 , t= 0.4 , F1: 0.891901242468
C= 0.1 , t= 0.5 , F1: 0.884885409395
C= 1 , t= 0.5 , F1: 0.893655552075
C= 10 , t= 0.5 , F1: 0.89739878903


## Seleção de features: n=3

In [39]:
my_features = []

#for i in range(bd1['channel'].unique().shape[0]):
#    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [40]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.852206166951
C= 1 , t= 0.3 , F1: 0.875198658117
C= 10 , t= 0.3 , F1: 0.874814868998
C= 0.1 , t= 0.4 , F1: 0.883848588799
C= 1 , t= 0.4 , F1: 0.887971286334
C= 10 , t= 0.4 , F1: 0.890514047912
C= 0.1 , t= 0.5 , F1: 0.879499511059
C= 1 , t= 0.5 , F1: 0.89082390849
C= 10 , t= 0.5 , F1: 0.890680360809


In [41]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
#for i in range(bd1['app'].unique().shape[0]):
#    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [42]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.856762587673
C= 1 , t= 0.3 , F1: 0.878810155006
C= 10 , t= 0.3 , F1: 0.878494632525
C= 0.1 , t= 0.4 , F1: 0.882212566716
C= 1 , t= 0.4 , F1: 0.886206912653
C= 10 , t= 0.4 , F1: 0.885059819467
C= 0.1 , t= 0.5 , F1: 0.859746409154
C= 1 , t= 0.5 , F1: 0.883540678038
C= 10 , t= 0.5 , F1: 0.889531226978


In [43]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd1['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [44]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.874427426041
C= 1 , t= 0.3 , F1: 0.883303104104
C= 10 , t= 0.3 , F1: 0.883584548379
C= 0.1 , t= 0.4 , F1: 0.889298627929
C= 1 , t= 0.4 , F1: 0.891643575843
C= 10 , t= 0.4 , F1: 0.893411801879
C= 0.1 , t= 0.5 , F1: 0.888601297532
C= 1 , t= 0.5 , F1: 0.891738896211
C= 10 , t= 0.5 , F1: 0.896472332957


In [45]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
#for i in range(4):
#    my_features.append('dia' + str(i))

In [46]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.877487341965
C= 1 , t= 0.3 , F1: 0.886636572133
C= 10 , t= 0.3 , F1: 0.886772226027
C= 0.1 , t= 0.4 , F1: 0.889276701557
C= 1 , t= 0.4 , F1: 0.892156512949
C= 10 , t= 0.4 , F1: 0.894758914528
C= 0.1 , t= 0.5 , F1: 0.884736694263
C= 1 , t= 0.5 , F1: 0.893124113011
C= 10 , t= 0.5 , F1: 0.8960029129


In [37]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
#for i in range(4):
#    my_features.append('dia' + str(i))

In [38]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.87779314647
C= 1 , t= 0.3 , F1: 0.886765884374
C= 10 , t= 0.3 , F1: 0.886198203427
C= 0.1 , t= 0.4 , F1: 0.891099517575
C= 1 , t= 0.4 , F1: 0.89406371732
C= 10 , t= 0.4 , F1: 0.894241934118
C= 0.1 , t= 0.5 , F1: 0.884489647777
C= 1 , t= 0.5 , F1: 0.89296166968
C= 10 , t= 0.5 , F1: 0.895250268302


In [27]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
#for i in range(4):
#    my_features.append('dia' + str(i))

In [28]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.3 , F1: 0.878574956859
C= 1 , t= 0.3 , F1: 0.886552357654
C= 10 , t= 0.3 , F1: 0.886345113669
C= 0.1 , t= 0.4 , F1: 0.891345055825
C= 1 , t= 0.4 , F1: 0.894598486719
C= 10 , t= 0.4 , F1: 0.893634717813
C= 0.1 , t= 0.5 , F1: 0.884550900276
C= 1 , t= 0.5 , F1: 0.893748674572
C= 10 , t= 0.5 , F1: 0.894421034707


## BD2

In [60]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

In [61]:
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

In [62]:
x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [64]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.106666666667
C= 10 , t= 0.2 , F1: 0.191626794258
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.199648660518
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.192418300654


# Seleção de features

In [65]:
my_features = []

#for i in range(bd2['channel'].unique().shape[0]):
#    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [66]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.0
C= 10 , t= 0.2 , F1: 0.0747826086957
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.0421052631579
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.05


In [67]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
#for i in range(bd2['app'].unique().shape[0]):
#    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [68]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.0877192982456
C= 10 , t= 0.2 , F1: 0.146602617655
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.120874493927
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.083422459893


In [69]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [70]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.190952380952
C= 10 , t= 0.2 , F1: 0.240173160173
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0333333333333
C= 10 , t= 0.3 , F1: 0.216919191919
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.179201680672


In [71]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [72]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.101724900487
C= 10 , t= 0.2 , F1: 0.210574162679
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.1
C= 10 , t= 0.3 , F1: 0.221903242956
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.221903242956


In [73]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd2['ip'].unique().shape[0]):
#    if(bd2[bd2['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [74]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.0952380952381
C= 10 , t= 0.2 , F1: 0.152629399586
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.152528180354
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.184103574444


In [75]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
#for i in range(4):
#    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [76]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.106666666667
C= 10 , t= 0.2 , F1: 0.199648660518
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.20202020202
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.194523563811


## Seleção n=4

In [77]:
my_features = []

#for i in range(bd2['channel'].unique().shape[0]):
#    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [78]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.0
C= 10 , t= 0.2 , F1: 0.141843364032
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.04
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.0533333333333


In [79]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
#for i in range(bd2['app'].unique().shape[0]):
#    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [80]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.0
C= 10 , t= 0.2 , F1: 0.140407177364
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.135353535354
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.0806722689076


In [81]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [82]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.212307692308
C= 10 , t= 0.2 , F1: 0.244334448161
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0888888888889
C= 10 , t= 0.3 , F1: 0.253725490196
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.253725490196


In [83]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd2['ip'].unique().shape[0]):
#    if(bd2[bd2['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [84]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.204871794872
C= 10 , t= 0.2 , F1: 0.18188034188
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0333333333333
C= 10 , t= 0.3 , F1: 0.178596491228
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.162418300654


In [85]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
#for i in range(4):
#    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [86]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.182142857143
C= 10 , t= 0.2 , F1: 0.188646125117
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0615384615385
C= 10 , t= 0.3 , F1: 0.179201680672
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.182142857143


## Seleção n=3

In [87]:
my_features = []

#for i in range(bd2['channel'].unique().shape[0]):
#    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [88]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.0
C= 10 , t= 0.2 , F1: 0.149860712387
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.08
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.0


In [89]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
#for i in range(bd2['app'].unique().shape[0]):
#    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [90]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.0
C= 10 , t= 0.2 , F1: 0.214902850772
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.0
C= 10 , t= 0.3 , F1: 0.153531379107
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.0


In [91]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd2['ip'].unique().shape[0]):
#    if(bd2[bd2['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [92]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.212307692308
C= 10 , t= 0.2 , F1: 0.214913202739
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.106060606061
C= 10 , t= 0.3 , F1: 0.210994152047
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.253725490196


In [93]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
#for i in range(4):
#    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [94]:
t = [0.2, 0.3, 0.4]
peso = [0.1, 1, 10]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = LogisticRegression(penalty='l2', C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = LogisticRegression(penalty='l2', C=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 0.1 , t= 0.2 , F1: 0.0
C= 1 , t= 0.2 , F1: 0.212307692308
C= 10 , t= 0.2 , F1: 0.244334448161
C= 0.1 , t= 0.3 , F1: 0.0
C= 1 , t= 0.3 , F1: 0.212307692308
C= 10 , t= 0.3 , F1: 0.235347985348
C= 0.1 , t= 0.4 , F1: 0.0
C= 1 , t= 0.4 , F1: 0.0
C= 10 , t= 0.4 , F1: 0.238681318681


In [103]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['output'].values


classifier = LogisticRegression(penalty='l2', C=10)
classifier.fit(x_train, y_train.ravel())

# predict class labels for the test set
predicted = classifier.predict(x_test)
probs = classifier.predict_proba(x_test)


In [104]:
pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i][1]) > 0.3):
        pred[i] = 1
    else:
        pred[i] = 0

In [105]:
print metrics.accuracy_score(y_test, pred)
print metrics.roc_auc_score(y_test, probs[:, 1])
print metrics.precision_score(y_test, pred)
print metrics.recall_score(y_test, pred)
print metrics.f1_score(y_test, pred)

0.997187420097
0.862838931207
0.583333333333
0.538461538462
0.56


# Árvore de Decisão

## BD1

In [52]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

((15643, 2998), (3911, 2998))


In [53]:
x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [54]:
t = [0.3, 0.4, 0.5]
peso = [10, 100, 1000]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 10 , t= 0.3 , F1: 0.873441991299
C= 100 , t= 0.3 , F1: 0.869942733434
C= 1000 , t= 0.3 , F1: 0.868469429808
C= 10 , t= 0.4 , F1: 0.873441991299
C= 100 , t= 0.4 , F1: 0.872513244424
C= 1000 , t= 0.4 , F1: 0.872663600498
C= 10 , t= 0.5 , F1: 0.872920802474
C= 100 , t= 0.5 , F1: 0.875291109251
C= 1000 , t= 0.5 , F1: 0.875609910642


In [55]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [56]:
t = [0.3, 0.4, 0.5]
peso = [10, 100, 1000]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 10 , t= 0.3 , F1: 0.873163303529
C= 100 , t= 0.3 , F1: 0.866693196317
C= 1000 , t= 0.3 , F1: 0.865478893881
C= 10 , t= 0.4 , F1: 0.874555344672
C= 100 , t= 0.4 , F1: 0.881452552834
C= 1000 , t= 0.4 , F1: 0.883014368531
C= 10 , t= 0.5 , F1: 0.874316782603
C= 100 , t= 0.5 , F1: 0.885184816267
C= 1000 , t= 0.5 , F1: 0.884424978258


In [57]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

clf = tree.DecisionTreeClassifier(max_depth=100)
clf = clf.fit(x_train, y_train)

x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['output'].values

probs = clf.predict_proba(x_test)

pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i][1]) > 0.5):
        pred[i] = 1
    else:
        pred[i] = 0
        
print metrics.accuracy_score(y_test, pred)
print metrics.roc_auc_score(y_test, probs[:, 1])
print metrics.precision_score(y_test, pred)
print metrics.recall_score(y_test, pred)
print metrics.f1_score(y_test, pred)

0.909486064945
0.946156229897
0.949971894323
0.864450127877
0.905195500803


## BD2

In [106]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)
    
x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

((15643, 3623), (3911, 3623))


In [107]:
t = [0.2, 0.3, 0.4]
peso = [10, 100, 1000]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 10 , t= 0.2 , F1: 0.199682539683
C= 100 , t= 0.2 , F1: 0.217039964866
C= 1000 , t= 0.2 , F1: 0.214141414141
C= 10 , t= 0.3 , F1: 0.183116883117
C= 100 , t= 0.3 , F1: 0.179545454545
C= 1000 , t= 0.3 , F1: 0.224500645553
C= 10 , t= 0.4 , F1: 0.223664369014
C= 100 , t= 0.4 , F1: 0.227735152032
C= 1000 , t= 0.4 , F1: 0.211111111111


In [108]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)
    
x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [109]:
t = [0.2, 0.3, 0.4]
peso = [10, 100, 1000]

for z in range(len(t)):

    xo = x2
    xo = xo.append(x3)
    xo = xo.append(x4)
    xo = xo.append(x5)

    yo = x1

    x_train = xo[my_features].values.reshape(-1,len(my_features))
    y_train = xo['output'].values.reshape(-1,1)

    for j in range(len(peso)):
    
        classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict_proba(x_test)
        tt = [0, 0, 0]
        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i][1]) > t[z]):
                pred[i] = 1
            else:
                pred[i] = 0
        tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
    
        for w in range(1,5):
            xo = x1
            if(w == 1):
                yo = x2
            else:
                xo.append(x2)
            if(w == 2):
                yo = x3
            else:
                xo.append(x3)
            if(w == 3):
                yo = x4
            else:
                xo.append(x4)
            if(w == 4):
                yo = x5
            else:
                xo.append(x5)

            x_train = xo[my_features].values.reshape(-1,len(my_features))
            y_train = xo['output'].values.reshape(-1,1)

            classifier = tree.DecisionTreeClassifier(max_depth=peso[j])
            classifier.fit(x_train, y_train.ravel())

            x_test = yo[my_features].values.reshape(-1,len(my_features))
            y_test = yo['output'].values

            probs = classifier.predict_proba(x_test)

            pred = [0]*probs.shape[0]
            for i in range(probs.shape[0]):
                if((probs[i][1]) > t[z]):
                    pred[i] = 1
                else:
                    pred[i] = 0
            tt[z] =  tt[z] + metrics.f1_score(y_test, pred)
        print 'C=', peso[j], ', t=', t[z], ', F1:', tt[z]*1.0/5


C= 10 , t= 0.2 , F1: 0.273414304993
C= 100 , t= 0.2 , F1: 0.246446886447
C= 1000 , t= 0.2 , F1: 0.248351648352
C= 10 , t= 0.3 , F1: 0.239763177998
C= 100 , t= 0.3 , F1: 0.20614973262
C= 1000 , t= 0.3 , F1: 0.209786096257
C= 10 , t= 0.4 , F1: 0.213389551625
C= 100 , t= 0.4 , F1: 0.206624040921
C= 1000 , t= 0.4 , F1: 0.206624040921


In [110]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))

x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

clf = tree.DecisionTreeClassifier(max_depth=10)
clf = clf.fit(x_train, y_train)

x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['output'].values

probs = clf.predict_proba(x_test)

pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i][1]) > 0.2):
        pred[i] = 1
    else:
        pred[i] = 0
        
print metrics.accuracy_score(y_test, pred)
print metrics.roc_auc_score(y_test, probs[:, 1])
print metrics.precision_score(y_test, pred)
print metrics.recall_score(y_test, pred)
print metrics.f1_score(y_test, pred)

0.994374840194
0.690472431622
0.263157894737
0.384615384615
0.3125


# SVM

## BD1

In [9]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd1['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
    
a = 0
for i in range(bd1['ip'].unique().shape[0]):
    if(bd1[bd1['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

((15643, 2998), (3911, 2998))


In [10]:
t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

xo = x2
xo = xo.append(x3)
xo = xo.append(x4)
xo = xo.append(x5)

yo = x1

x_train = xo[my_features].values.reshape(-1,len(my_features))
y_train = xo['output'].values.reshape(-1,1)

tt = [0, 0, 0]


for j in range(len(peso)):
    now = datetime.now()
    print now.minute
    classifier = SVC(C=peso[j])
    classifier.fit(x_train, y_train.ravel())
    now = datetime.now()
    print now.minute

    x_test = yo[my_features].values.reshape(-1,len(my_features))
    y_test = yo['output'].values
    
    probs = classifier.predict(x_test)
    pred = [0]*probs.shape[0]
    for i in range(probs.shape[0]):
        if((probs[i]) > 0):
            pred[i] = 1
    tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    
    for w in range(1,5):
        xo = x1
        if(w == 1):
            yo = x2
        else:
            xo.append(x2)
        if(w == 2):
            yo = x3
        else:
            xo.append(x3)
        if(w == 3):
            yo = x4
        else:
            xo.append(x4)
        if(w == 4):
            yo = x5
        else:
            xo.append(x5)

        x_train = xo[my_features].values.reshape(-1,len(my_features))
        y_train = xo['output'].values.reshape(-1,1)

        classifier = SVC(C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict(x_test)

        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i]) > 0):
                pred[i] = 1
        tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    print 'C=', peso[j], ', F1:', tt[j]*1.0/5


23
35
C= 0.1 , F1: 0.632352151587
43
44
C= 1 , F1: 0.713350935471
49
50
C= 10 , F1: 0.870180932631


In [11]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [12]:
t = [0.3, 0.4, 0.5]
peso = [0.1, 1, 10]

xo = x2
xo = xo.append(x3)
xo = xo.append(x4)
xo = xo.append(x5)

yo = x1

x_train = xo[my_features].values.reshape(-1,len(my_features))
y_train = xo['output'].values.reshape(-1,1)

tt = [0, 0, 0]


for j in range(len(peso)):
    now = datetime.now()
    print now.minute
    classifier = SVC(C=peso[j])
    classifier.fit(x_train, y_train.ravel())
    now = datetime.now()
    print now.minute

    x_test = yo[my_features].values.reshape(-1,len(my_features))
    y_test = yo['output'].values
    
    probs = classifier.predict(x_test)
    pred = [0]*probs.shape[0]
    for i in range(probs.shape[0]):
        if((probs[i]) > 0):
            pred[i] = 1
    tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    
    for w in range(1,5):
        xo = x1
        if(w == 1):
            yo = x2
        else:
            xo.append(x2)
        if(w == 2):
            yo = x3
        else:
            xo.append(x3)
        if(w == 3):
            yo = x4
        else:
            xo.append(x4)
        if(w == 4):
            yo = x5
        else:
            xo.append(x5)

        x_train = xo[my_features].values.reshape(-1,len(my_features))
        y_train = xo['output'].values.reshape(-1,1)

        classifier = SVC(C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict(x_test)

        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i]) > 0):
                pred[i] = 1
        tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    print 'C=', peso[j], ', F1:', tt[j]*1.0/5


53
0
C= 0.1 , F1: 0.632352151587
5
5
C= 1 , F1: 0.785797242903
9
9
C= 10 , F1: 0.874358223783


In [13]:
my_features = []

for i in range(bd1['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd1['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd1['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

#for i in range(bd1['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
#a = 0
#for i in range(bd1['ip'].unique().shape[0]):
#    if(bd1[bd1['ip']==i].shape[0] > 1):
#        my_features.append('ip' + str(a))
#        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

clf = SVC(C=10)
clf = clf.fit(x_train, y_train.ravel())

x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['output'].values.reshape(-1,1)

probs = clf.predict(x_test)

pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i]) > 0):
        pred[i] = 1
        
print metrics.accuracy_score(y_test, pred)
print metrics.precision_score(y_test, pred)
print metrics.recall_score(y_test, pred)
print metrics.f1_score(y_test, pred)

0.909997443109
0.973979893554
0.842455242967
0.903455842019


## BD2

In [14]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
for i in range(bd2['device'].unique().shape[0]):
    my_features.append('device' + str(i))    

for i in range(bd2['os'].unique().shape[0]):
    my_features.append('os' + str(i))
    
for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1    
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

((15643, 3623), (3911, 3623))


In [15]:
t = [0.2, 0.3, 0.4]
peso = [10, 100, 1000]

xo = x2
xo = xo.append(x3)
xo = xo.append(x4)
xo = xo.append(x5)

yo = x1

x_train = xo[my_features].values.reshape(-1,len(my_features))
y_train = xo['output'].values.reshape(-1,1)

tt = [0, 0, 0]


for j in range(len(peso)):
    now = datetime.now()
    print now.minute
    classifier = SVC(C=peso[j])
    classifier.fit(x_train, y_train.ravel())
    now = datetime.now()
    print now.minute

    x_test = yo[my_features].values.reshape(-1,len(my_features))
    y_test = yo['output'].values
    
    probs = classifier.predict(x_test)
    pred = [0]*probs.shape[0]
    for i in range(probs.shape[0]):
        if((probs[i]) > 0):
            pred[i] = 1
    tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    
    for w in range(1,5):
        xo = x1
        if(w == 1):
            yo = x2
        else:
            xo.append(x2)
        if(w == 2):
            yo = x3
        else:
            xo.append(x3)
        if(w == 3):
            yo = x4
        else:
            xo.append(x4)
        if(w == 4):
            yo = x5
        else:
            xo.append(x5)

        x_train = xo[my_features].values.reshape(-1,len(my_features))
        y_train = xo['output'].values.reshape(-1,1)

        classifier = SVC(C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict(x_test)

        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i]) > 0):
                pred[i] = 1
        tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    print 'C=', peso[j], ', F1:', tt[j]*1.0/5


4
4
C= 10 , F1: 0.0
4
4
C= 100 , F1: 0.0
4
4
C= 1000 , F1: 0.204823246929


In [16]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)
    
x1 = train_data.iloc[0:int(x_train.shape[0]/5),:]
x2 = train_data.iloc[int(x_train.shape[0]/5):int(2*x_train.shape[0]/5),:]
x3 = train_data.iloc[int(2*x_train.shape[0]/5):int(3*x_train.shape[0]/5),:]
x4 = train_data.iloc[int(3*x_train.shape[0]/5):int(4*x_train.shape[0]/5),:]
x5 = train_data.iloc[int(4*x_train.shape[0]/5):x_train.shape[0],:]

In [17]:
t = [0.2, 0.3, 0.4]
peso = [10, 100, 1000]

xo = x2
xo = xo.append(x3)
xo = xo.append(x4)
xo = xo.append(x5)

yo = x1

x_train = xo[my_features].values.reshape(-1,len(my_features))
y_train = xo['output'].values.reshape(-1,1)

tt = [0, 0, 0]


for j in range(len(peso)):
    now = datetime.now()
    print now.minute
    classifier = SVC(C=peso[j])
    classifier.fit(x_train, y_train.ravel())
    now = datetime.now()
    print now.minute

    x_test = yo[my_features].values.reshape(-1,len(my_features))
    y_test = yo['output'].values
    
    probs = classifier.predict(x_test)
    pred = [0]*probs.shape[0]
    for i in range(probs.shape[0]):
        if((probs[i]) > 0):
            pred[i] = 1
    tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    
    for w in range(1,5):
        xo = x1
        if(w == 1):
            yo = x2
        else:
            xo.append(x2)
        if(w == 2):
            yo = x3
        else:
            xo.append(x3)
        if(w == 3):
            yo = x4
        else:
            xo.append(x4)
        if(w == 4):
            yo = x5
        else:
            xo.append(x5)

        x_train = xo[my_features].values.reshape(-1,len(my_features))
        y_train = xo['output'].values.reshape(-1,1)

        classifier = SVC(C=peso[j])
        classifier.fit(x_train, y_train.ravel())

        x_test = yo[my_features].values.reshape(-1,len(my_features))
        y_test = yo['output'].values

        probs = classifier.predict(x_test)

        pred = [0]*probs.shape[0]
        for i in range(probs.shape[0]):
            if((probs[i]) > 0):
                pred[i] = 1
        tt[j] =  tt[j] + metrics.f1_score(y_test, pred)
    print 'C=', peso[j], ', F1:', tt[j]*1.0/5


5
5
C= 10 , F1: 0.0
5
5
C= 100 , F1: 0.0
6
6
C= 1000 , F1: 0.232406808877


In [11]:
my_features = []

for i in range(bd2['channel'].unique().shape[0]):
    my_features.append('channel' + str(i))

    
for i in range(bd2['app'].unique().shape[0]):
    my_features.append('app' + str(i))
    
#for i in range(bd2['device'].unique().shape[0]):
#    my_features.append('device' + str(i))    

#for i in range(bd2['os'].unique().shape[0]):
#    my_features.append('os' + str(i))
    
a = 0
for i in range(bd2['ip'].unique().shape[0]):
    if(bd2[bd2['ip']==i].shape[0] > 1):
        my_features.append('ip' + str(a))
        a = a + 1
    
for i in range(4):
    my_features.append('dia' + str(i))
    
x_train = train_data[my_features].values.reshape(-1,len(my_features))
y_train = train_data['output'].values.reshape(-1,1)

clf = SVC(C=1000)
clf = clf.fit(x_train, y_train.ravel())

x_test = test_data[my_features].values.reshape(-1,len(my_features))
y_test = test_data['output'].values.reshape(-1,1)

probs = clf.predict(x_test)

pred = [0]*probs.shape[0]
for i in range(probs.shape[0]):
    if((probs[i]) > 0):
        pred[i] = 1
        
print metrics.accuracy_score(y_test, pred)
print metrics.precision_score(y_test, pred)
print metrics.recall_score(y_test, pred)
print metrics.f1_score(y_test, pred)

0.996931731015
0.571428571429
0.307692307692
0.4


# Baseline 1

## BD1

In [15]:
test_data = test_data.sample(frac=1, random_state=200).reset_index(drop=True)

In [9]:
cont = 0
recall = 0
acuracia = 0
prev = 1
fpositive = 0
fnegative = 0
tpositive = 0
tnegative = 0

if(test_data[test_data['output']==1].shape[0]*1.0/test_data.shape[0] < 0.5):
    prev = 0

for i in range(test_data.shape[0]):
    if(test_data.output[i] == prev):
        cont = cont + 1

acuracia = cont*1.0/test_data.shape[0]
precision = acuracia
recall = 1

fpositive = test_data.shape[0] - cont
tpositive = cont
tnegative = 0
fnegative = 0

if(prev == 0):
    fpositive = 0
    tpositive = 0
    tnegative = cont
    fnegative = test_data.shape[0] - cont
    precision = 0
    recall = 0

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall
print 'true positive:', tpositive
print 'false positive:', fpositive
print 'true negative:', tnegative
print 'false negative:', fnegative

acurácia: 0.500127844541
precisão: 0
revocação: 0
true positive: 0
false positive: 0
true negative: 1956
false negative: 1955


## BD2

In [16]:
cont = 0
recall = 0
acuracia = 0
prev = 1
fpositive = 0
fnegative = 0
tpositive = 0
tnegative = 0

if(test_data[test_data['output']==1].shape[0]*1.0/test_data.shape[0] < 0.5):
    prev = 0

for i in range(test_data.shape[0]):
    if(test_data.output[i] == prev):
        cont = cont + 1

acuracia = cont*1.0/test_data.shape[0]
precision = acuracia
recall = 1

fpositive = test_data.shape[0] - cont
tpositive = cont
tnegative = 0
fnegative = 0

if(prev == 0):
    fpositive = 0
    tpositive = 0
    tnegative = cont
    fnegative = test_data.shape[0] - cont
    precision = 0
    recall = 0

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall
print 'true positive:', tpositive
print 'false positive:', fpositive
print 'true negative:', tnegative
print 'false negative:', fnegative

acurácia: 0.996676041933
precisão: 0
revocação: 0
true positive: 0
false positive: 0
true negative: 3898
false negative: 13


# Baseline 2 (aleatório)

## BD1

In [12]:
acuracia = 0
recall = 0
precision = 0
fpositive = 0
fnegative = 0
tpositive = 0
tnegative = 0

for i in range(test_data.shape[0]):
    aleatorio = randint(0,1)
    if(aleatorio == 1):
        precision = precision + 1
        if(test_data.output[i] == aleatorio):
            acuracia = acuracia + 1
            recall = recall + 1
            tpositive = tpositive + 1
        else:
            fpositive = fpositive + 1
    elif(test_data.output[i] == aleatorio):
        acuracia = acuracia + 1
        tnegative = tnegative + 1
    else:
        fnegative = fnegative + 1

acuracia = acuracia*1.0/test_data.shape[0]
precision = recall*1.0/precision
recall = recall*1.0/test_data[test_data['output'] == 1].shape[0]

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall
print 'true positive:', tpositive
print 'false positive:', fpositive
print 'true negative:', tnegative
print 'false negative:', fnegative

acurácia: 0.509076962414
precisão: 0.508878741755
revocação: 0.513043478261
true positive: 1003
false positive: 968
true negative: 988
false negative: 952


## BD2

In [18]:
acuracia = 0
recall = 0
precision = 0
fpositive = 0
fnegative = 0
tpositive = 0
tnegative = 0

for i in range(test_data.shape[0]):
    aleatorio = randint(0,1)
    if(aleatorio == 1):
        precision = precision + 1
        if(test_data.output[i] == aleatorio):
            acuracia = acuracia + 1
            recall = recall + 1
            tpositive = tpositive + 1
        else:
            fpositive = fpositive + 1
    elif(test_data.output[i] == aleatorio):
        acuracia = acuracia + 1
        tnegative = tnegative + 1
    else:
        fnegative = fnegative + 1

acuracia = acuracia*1.0/test_data.shape[0]
precision = recall*1.0/precision
recall = recall*1.0/test_data[test_data['output'] == 1].shape[0]

print 'acurácia:', acuracia
print 'precisão:', precision
print 'revocação:', recall
print 'true positive:', tpositive
print 'false positive:', fpositive
print 'true negative:', tnegative
print 'false negative:', fnegative

acurácia: 0.485553566863
precisão: 0.00347739692002
revocação: 0.538461538462
true positive: 7
false positive: 2006
true negative: 1892
false negative: 6
